<a href="https://colab.research.google.com/github/olucasaguiar/estudos-sobre-machine-learning/blob/main/temas/natural-language-processing/NLP_lucas_nascimento_aguiar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Sobre o dataset

O UTLCorpus, proveniente do projeto Opinando, é um corpus com mais de 2 milhões de avaliações. Ele engloba resenhas de filmes coletadas do Filmow, uma popular rede social de filmes, e foi tratado especificamente como UTLC-Movies neste contexto, com todas as avaliações que possuíam classificação 0 removidas para garantir a qualidade dos dados.

In [1]:
# @title Carregando o dataset
import kagglehub
import pandas as pd
from kagglehub import KaggleDatasetAdapter


def load_dataset(handle: str, path: str) -> pd.DataFrame:
    return kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        handle,
        path,
    )

In [2]:
import numpy as np


dataset_handle = "fredericods/ptbr-sentiment-analysis-datasets"
dataset_file = "utlc_movies.csv"

df = load_dataset(handle=dataset_handle, path=dataset_file)
df = df[["review_text", "polarity", "rating"]]

### Reduzi o tamanho do dataset para acelerar o treinamento do modelo
TEST_SIZE = 10_000
### Para garantir a mesma proporção do conjunto de testes para classificação 1/0
FOLD_SIZE = TEST_SIZE // 2
positive_samples = df[df["polarity"] == 1].sample(
    n=FOLD_SIZE,
    random_state=42,
)
negative_samples = df[df["polarity"] == 0].sample(
    n=FOLD_SIZE,
    random_state=42,
)

df = (
    pd.concat([positive_samples, negative_samples], ignore_index=True)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)
print(df.info())
print(df.head())

Using Colab cache for faster access to the 'ptbr-sentiment-analysis-datasets' dataset.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   review_text  10000 non-null  object 
 1   polarity     10000 non-null  float64
 2   rating       10000 non-null  int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 234.5+ KB
None
                                         review_text  polarity  rating
0  Horrível! Filme cansativo, arrastado, com ator...       0.0       1
1  Sem dúvidas posso afirmar que é um dos melhore...       1.0       5
2                                            Ameii!!       1.0       5
3  Não desmerecendo as outras atrizes, mas eu fic...       1.0       4
4  Maravilhoso! Eu juro q não dava nada pelo film...       1.0       5


In [3]:
# @title Preparando o dataset
import nltk
import pandas as pd
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

TrainSet = tuple[pd.DataFrame, pd.Series]
TestSet = tuple[pd.DataFrame, pd.Series]


def prepare_dataset(
    feature_values: pd.DataFrame | pd.Series,
    target_values: pd.DataFrame | pd.Series,
    max_features: int | None = None,
    max_df: float = 1.0,
    min_df: int | float = 1,
    norm: str | None = "l2",
) -> tuple[TfidfVectorizer, TrainSet, TestSet]:
    # step 1: Split test and train data
    X_train, X_test, y_train, y_test = train_test_split(
        feature_values, target_values, test_size=0.33, random_state=42, shuffle=True
    )

    # step 2: Create TF-IDF Vectorizer for documents vocabulary
    nltk.download("stopwords")
    vectorizer = TfidfVectorizer(
        strip_accents="ascii",
        stop_words=stopwords.words("portuguese"),
        norm=norm,
        max_df=max_df,
        min_df=min_df,
        max_features=max_features,
    )

    # step 3: Fit data into vetorizer/tokenizer and transform feature_data
    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)

    # step 4: Return train and test datasets
    return vectorizer, (X_train, y_train), (X_test, y_test)

In [4]:
vectorizer, (X_train, y_train), (X_test, y_test) = prepare_dataset(
    feature_values=df["review_text"], target_values=df["polarity"]
)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ate', 'eramos', 'estao', 'estavamos', 'estiveramos', 'estivessemos', 'foramos', 'fossemos', 'ha', 'hao', 'houveramos', 'houverao', 'houveriamos', 'houvessemos', 'ja', 'nao', 'sao', 'sera', 'serao', 'seriamos', 'so', 'tambem', 'tera', 'terao', 'teriamos', 'tinhamos', 'tiveramos', 'tivessemos', 'voce', 'voces'] not in stop_words.
  warnings.warn(


In [5]:
import numpy as np

print("| qtd. documentos | tam. vocabulario |")
print("| --------------- | ---------------- |")
print("| %15d | %16d |" % X_train.shape)

| qtd. documentos | tam. vocabulario |
| --------------- | ---------------- |
|            6700 |            19492 |


In [6]:
# @title Aplicando o treinamento do modelo supervisionado
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix


clf = RandomForestClassifier()
# clf = MLPClassifier()

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Matriz de Confusão:")
print(confusion_matrix(y_test, y_pred))

print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred))

Matriz de Confusão:
[[1231  389]
 [ 372 1308]]

Relatório de Classificação:
              precision    recall  f1-score   support

         0.0       0.77      0.76      0.76      1620
         1.0       0.77      0.78      0.77      1680

    accuracy                           0.77      3300
   macro avg       0.77      0.77      0.77      3300
weighted avg       0.77      0.77      0.77      3300



In [7]:
# @title Gerando dados simulados com LLM
from google import genai
from google.colab import userdata
from google.genai import types


def generate_content(
    prompt: str,
    model: str = "gemini-2.5-flash-lite",
    system_instruction: str | None = None,
    temperature: float | None = None,
    top_p: float | None = None,
) -> str:
    try:
        genai_api_key = userdata.get("GOOGLE_API_KEY")
    except userdata.SecretNotFoundError as e:
        print(
            'Secret not found. Configure secret for Gemini Models using secret variable "GOOGLE_API_KEY".'
        )
        raise e

    with genai.Client(api_key=genai_api_key) as client:
        response = client.models.generate_content(
            model=model,
            contents=types.Part.from_text(text=prompt),
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=temperature,
                top_p=top_p,
            ),
        )
        return response.text

In [9]:
from io import StringIO

import pandas as pd
from sklearn.utils import shuffle

few_shot_prompt = """
Prepare um conjunto de dados estruturados no formato `.csv` no idioma português (pt_BR) com as dimensões:
- **index**: Valor numérico sequêncial iniciando em 1;
- **review_text**: Um texto entre 5 e 25 palavras contendo a opinião sobre um filme qualquer;
- **polarity**: A intenção do comentário, onde 0 representa negativo e 1 representa positivo.

Cada linha deve simular a avaliação de uma pessoa diferente, com suas características únicas e pessoais.
O público respondente possui faixa etária de 20 à 45 anos, e variação demográfica (genêro, contexto social, crença, etc).

**Exemplos**

Usuário: Prepare um conjunto de 5 analises positivas e 5 análises negativas
Resposta:
```csv
index,review_text,polarity
1,"Esse filme me deixou maravilhado, a história é envolvente e os atores são incríveis. Uma obra-prima!",1
2,"Adorei cada segundo! A fotografia é deslumbrante e a trilha sonora emocionou demais. Recomendo muito!",1
3,"Que experiência cinematográfica fantástica! Os efeitos visuais são de outro mundo e a trama me prendeu do início ao fim.",1
4,"Um filme que aquece o coração. A mensagem é poderosa e me fez refletir bastante sobre a vida e o amor.",1
5,"Simplesmente sensacional! Roteiro inteligente, atuações impecáveis e uma direção que me transportou para outro universo.",1
6,"Que decepção profunda! A história era confusa e o final me deixou com um gosto amargo na boca. Não gostei.",0
7,"Um desperdício de tempo. O filme foi arrastado, sem emoção e com diálogos rasos. Não recomendo de forma alguma.",0
8,"Odiei a atuação dos personagens principais. Pareciam forçados e sem carisma algum. Um filme muito fraco.",0
9,"Achei o enredo previsível e sem originalidade. Faltou criatividade e surpreendentes reviravoltas. Muito meh.",0
10,"Não curti nada neste filme. Os efeitos eram amadores, o som péssimo e a narrativa sem nexo. Paguei caro pra me frustrar.",0
```

Usuário: Prepare um conjunto de 3 analises positivas e 5 análises negativas
Resposta:
```csv
index,review_text,polarity
1,"Amei a atuação! Os personagens eram cativantes e a história me prendeu do começo ao fim. Um filme realmente inspirador.",1
2,"Que surpresa agradável! A fotografia era espetacular e a trilha sonora me tocou profundamente. Recomendo para todos!.",1
3,"Um filme que me fez rir e chorar. A mensagem sobre família e amizade é linda e muito bem contada.",1
4,"Que decepção! A trama parecia promissora, mas se perdeu em clichês e previsibilidade. Não valeu o ingresso.",0
5,"O pior filme que vi este ano. Atuações pífias e um roteiro sem pé nem cabeça. Arrependimento total.",0
6,"Não consegui me conectar com nenhum personagem. A história foi arrastada e o final me deixou frustrado e sem resposta.",0
7,"Um desastre cinematográfico. Diálogos forçados, efeitos visuais de quinta categoria e uma falta de originalidade gritante.",0
8,"Paguei caro para ver algo tão medíocre. A narrativa é confusa e não empolga em nenhum momento. Perda de tempo.",0
```

**Solicitação**
Usuário: Prepare um conjunto de {n_positive} análises positivas e {n_negative} análises negativas
Resposta:
""".strip()


def simulate_review(n_positive: int, n_negative: int) -> pd.DataFrame:
    prompt = few_shot_prompt.format(n_positive=n_positive, n_negative=n_negative)
    response = generate_content(prompt=prompt, model='gemini-3.1-flash-lite-preview', temperature=0.3, top_p=1.0)

    if response.startswith("```csv"):
        response = response[6:]

    if response.endswith("```"):
        response = response[:-3]

    buffer = StringIO(response)
    review_df = pd.read_csv(buffer)
    review_df = shuffle(review_df, random_state=42).reset_index(drop=True)
    return review_df[["review_text", "polarity"]]

In [10]:
llm_data = simulate_review(n_positive=50, n_negative=50)

print("| qtd. análises | qtd. positivos | qtd. negativos |")
print("| ------------- | -------------- | -------------- |")
print(
    "| %13d | %14d | %14d |"
    % (
        len(llm_data),
        len(llm_data[llm_data["polarity"] == 1]),
        len(llm_data[llm_data["polarity"] == 0]),
    )
)

TimeoutException: Requesting secret GOOGLE_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:
X_llm_test = vectorizer.transform(llm_data["review_text"])

y_llm_pred = clf.predict(X_llm_test)

print("Matriz de Confusão:")
print(confusion_matrix(llm_data["polarity"], y_llm_pred))

print("\nRelatório de Classificação:")
print(classification_report(llm_data["polarity"], y_llm_pred))